# 06 - Feature Analysis, Leakage Risk & Statistical Testing
**Project**: AI-Based Structural Variant Prioritization from Long-Read Sequencing Using Explainable Machine Learning  

## Objective
- Evaluate all candidate features for machine learning
- Statistical hypothesis testing (Mann-Whitney U, Chi-Square)
- Rigorous data leakage prevention analysis
- Final target formulation and model evaluation recommendations


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Automatically find file from either project root or eda/ folder
def resolve_path(p):
    for candidate in [p, os.path.join("..", p), os.path.join("data", p), os.path.join("..", "data", p)]:
        if os.path.exists(candidate):
            return candidate
    return p

data_path = resolve_path("processed/clinvar_sv_cleaned.csv.gz")
df_cv = pd.read_csv(data_path, low_memory=False)

def categorize_clinsig(cln):
    c = str(cln).lower()
    if "pathogenic" in c and "conflict" not in c:
        return "Pathogenic"
    elif "benign" in c and "conflict" not in c:
        return "Benign"
    elif "uncertain" in c or "vus" in c:
        return "VUS"
    else:
        return "Other"

df_cv["class"] = df_cv["clinsig"].apply(categorize_clinsig)
df_path = df_cv[df_cv["class"] == "Pathogenic"]
df_benign = df_cv[df_cv["class"] == "Benign"]

print(f"Loaded from: {data_path}")
print(f"Pathogenic SVs: {len(df_path):,}")
print(f"Benign SVs:     {len(df_benign):,}")


Loaded from: processed/clinvar_sv_cleaned.csv.gz
Pathogenic SVs: 45,277
Benign SVs:     13,306


## Statistical Comparison 1: SV Length
**Hypothesis**: Pathogenic SVs and Benign SVs have significantly different genomic length distributions.


In [2]:
mwu_stat, mwu_pval = stats.mannwhitneyu(df_path["length"], df_benign["length"])
print(f"Pathogenic Median Length: {df_path['length'].median():,.0f} bp")
print(f"Benign Median Length:     {df_benign['length'].median():,.0f} bp")
print(f"Mann-Whitney U p-value:    {mwu_pval:.4e} (Statistically Significant)")


Pathogenic Median Length: 51,100 bp
Benign Median Length:     153,420 bp
Mann-Whitney U p-value:    3.0389e-77 (Statistically Significant)


## Statistical Comparison 2: SV Type Distribution
**Hypothesis**: Structural variant types (Deletion, Duplication, CNV loss/gain) are non-randomly associated with clinical pathogenicity.


In [3]:
ct = pd.crosstab(df_cv[df_cv['class'].isin(['Pathogenic', 'Benign'])]['class'], df_cv['type'])
chi2, p_val, dof, _ = stats.chi2_contingency(ct)
print("Contingency Table:")
print(ct)
print(f"Chi-Square = {chi2:.2f}, p-value = {p_val:.4e}")


Contingency Table:
type        Complex  Deletion  Duplication  Indel  Insertion  Inversion  \
class                                                                     
Benign            1       624          207     14          0          0   
Pathogenic       32     23781         2481    739          4         48   

type        Microsatellite  Translocation  Variation  copy number gain  \
class                                                                    
Benign                  22              0          0              7579   
Pathogenic              17             24          1              5479   

type        copy number loss  fusion  
class                                 
Benign                  4859       0  
Pathogenic             12670       1  
Chi-Square = 15769.58, p-value = 0.0000e+00


## Candidate Feature Matrix & Leakage Analysis Table

In [4]:
feature_candidates = [
    {"Feature": "SV Length (bp)", "Source": "ClinVar / Caller", "Type": "Numeric", "Leakage Risk": "None", "Decision": "KEEP"},
    {"Feature": "SV Type (DEL, DUP, INV, INS, CNV)", "Source": "ClinVar / Caller", "Type": "Categorical", "Leakage Risk": "None", "Decision": "KEEP"},
    {"Feature": "Chromosome", "Source": "ClinVar / Caller", "Type": "Categorical", "Leakage Risk": "None", "Decision": "KEEP"},
    {"Feature": "gnomAD Global AF", "Source": "gnomAD SV", "Type": "Numeric", "Leakage Risk": "None", "Decision": "KEEP"},
    {"Feature": "gnomAD PopMax AF", "Source": "gnomAD SV", "Type": "Numeric", "Leakage Risk": "None", "Decision": "KEEP"},
    {"Feature": "gnomAD Subpopulation AFs (AFR, EUR, etc.)", "Source": "gnomAD SV", "Type": "Numeric", "Leakage Risk": "None", "Decision": "KEEP"},
    {"Feature": "Gene Overlap / Symbol", "Source": "ClinVar / RefSeq", "Type": "Categorical", "Leakage Risk": "Gene-memorization risk", "Decision": "REPLACE with Gene-level constraint (pLI, LOEUF)"},
    {"Feature": "Loss of Function (LoF) Flag", "Source": "gnomAD / Ensembl", "Type": "Binary", "Leakage Risk": "None", "Decision": "KEEP"},
    {"Feature": "Promoter / Regulatory Overlap", "Source": "gnomAD / ENCODE", "Type": "Binary", "Leakage Risk": "None", "Decision": "KEEP"},
    {"Feature": "ClinVar ReviewStatus (Stars)", "Source": "ClinVar", "Type": "Ordinal", "Leakage Risk": "HIGH LEAKAGE", "Decision": "DROP (Only use for sample weighting/filtering)"},
    {"Feature": "PhenotypeList / PhenotypeIDS", "Source": "ClinVar", "Type": "Text/ID", "Leakage Risk": "EXTREME LEAKAGE", "Decision": "DROP (Encodes target diagnosis directly)"},
    {"Feature": "NumberSubmitters", "Source": "ClinVar", "Type": "Numeric", "Leakage Risk": "HIGH LEAKAGE", "Decision": "DROP (Proxy for pathogenic clinical attention)"}
]

pd.DataFrame(feature_candidates)


,Feature,Source,Type,Leakage Risk,Decision
0,SV Length (bp),ClinVar / Caller,Numeric,None,KEEP
1,"SV Type (DEL, DUP, INV, INS, CNV)",ClinVar / Caller,Categorical,None,KEEP
2,Chromosome,ClinVar / Caller,Categorical,None,KEEP
3,gnomAD Global AF,gnomAD SV,Numeric,None,KEEP
4,gnomAD PopMax AF,gnomAD SV,Numeric,None,KEEP
5,"gnomAD Subpopulation AFs (AFR, EUR, etc.)",gnomAD SV,Numeric,None,KEEP
6,Gene Overlap / Symbol,ClinVar / RefSeq,Categorical,Gene-memorization risk,"REPLACE with Gene-level constraint (pLI, LOEUF)"
7,Loss of Function (LoF) Flag,gnomAD / Ensembl,Binary,None,KEEP
8,Promoter / Regulatory Overlap,gnomAD / ENCODE,Binary,None,KEEP
9,ClinVar ReviewStatus (Stars),ClinVar,Ordinal,HIGH LEAKAGE,DROP (Only use for sample weighting/filtering)
